In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    r"C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148"
)

FEATURE_FILE = (
    DATA_ROOT
    / "processed_windows"
    / "combined_eeg_features.csv"
)


# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

ENROLL_SESSIONS = [
    "ses-session1",
    "ses-session2"
]

VERIFY_SESSION = "ses-session3"

RANDOM_STATE = 42


# ============================================================
# LOAD FEATURES
# ============================================================

features_df = pd.read_csv(FEATURE_FILE)

print("Dataset shape:", features_df.shape)
print()

print("Sessions:")
print(features_df["session"].value_counts())

print()

print("Subjects:", features_df["subject"].nunique())

Dataset shape: (8735, 111)

Sessions:
session
ses-session1    3060
ses-session2    3040
ses-session3    2635
Name: count, dtype: int64

Subjects: 60


In [2]:
META_COLUMNS = [
    "subject",
    "session",
    "window"
]


ALL_FEATURE_COLUMNS = [
    col
    for col in features_df.columns
    if col not in META_COLUMNS
]


print("Total feature count:", len(ALL_FEATURE_COLUMNS))

print("\nFirst 20 features:")
for col in ALL_FEATURE_COLUMNS[:20]:
    print(col)

Total feature count: 108

First 20 features:
Fp1_delta_power
Fp1_delta_logpower
Fp1_theta_power
Fp1_theta_logpower
Fp1_alpha_power
Fp1_alpha_logpower
Fp1_beta_power
Fp1_beta_logpower
Fp1_gamma_power
Fp1_gamma_logpower
Fp1_total_logpower
Fp1_delta_relative
Fp1_delta_logrelative
Fp1_theta_relative
Fp1_theta_logrelative
Fp1_alpha_relative
Fp1_alpha_logrelative
Fp1_beta_relative
Fp1_beta_logrelative
Fp1_gamma_relative


In [3]:
# ============================================================
# ENROLLMENT
# ============================================================

enrollment_df = features_df[
    features_df["session"].isin(ENROLL_SESSIONS)
].copy()


# ============================================================
# VERIFICATION
# ============================================================

verification_df = features_df[
    features_df["session"] == VERIFY_SESSION
].copy()


print("Enrollment shape:", enrollment_df.shape)
print("Verification shape:", verification_df.shape)

print()

print("Enrollment sessions:")
print(enrollment_df["session"].value_counts())

print()

print("Verification sessions:")
print(verification_df["session"].value_counts())

print()

print(
    "Enrollment subjects:",
    enrollment_df["subject"].nunique()
)

print(
    "Verification subjects:",
    verification_df["subject"].nunique()
)

Enrollment shape: (6100, 111)
Verification shape: (2635, 111)

Enrollment sessions:
session
ses-session1    3060
ses-session2    3040
Name: count, dtype: int64

Verification sessions:
session
ses-session3    2635
Name: count, dtype: int64

Enrollment subjects: 60
Verification subjects: 57


In [4]:
# ============================================================
# COMPACT FEATURE SELECTION
# ============================================================

SELECTED_FEATURES = []


for col in ALL_FEATURE_COLUMNS:

    # --------------------------------------------------------
    # Log bandpower
    # --------------------------------------------------------

    if "_logpower" in col:
        SELECTED_FEATURES.append(col)

    # --------------------------------------------------------
    # Log relative bandpower
    # --------------------------------------------------------

    elif "_logrelative" in col:
        SELECTED_FEATURES.append(col)

    # --------------------------------------------------------
    # Frontal asymmetry
    # --------------------------------------------------------

    elif col.startswith("asymmetry_"):
        SELECTED_FEATURES.append(col)

    # --------------------------------------------------------
    # Channel log ratios
    # --------------------------------------------------------

    elif col.startswith("logratio_"):
        SELECTED_FEATURES.append(col)

    # --------------------------------------------------------
    # Hjorth
    # --------------------------------------------------------

    elif "_hjorth_" in col:
        SELECTED_FEATURES.append(col)


SELECTED_FEATURES = sorted(
    set(SELECTED_FEATURES)
)


print("Selected features:", len(SELECTED_FEATURES))

print("\nSelected features:")
for col in SELECTED_FEATURES:
    print(col)

Selected features: 54

Selected features:
Fp1_alpha_logpower
Fp1_alpha_logrelative
Fp1_beta_logpower
Fp1_beta_logrelative
Fp1_delta_logpower
Fp1_delta_logrelative
Fp1_gamma_logpower
Fp1_gamma_logrelative
Fp1_hjorth_complexity
Fp1_hjorth_mobility
Fp1_theta_logpower
Fp1_theta_logrelative
Fp1_total_logpower
Fp2_alpha_logpower
Fp2_alpha_logrelative
Fp2_beta_logpower
Fp2_beta_logrelative
Fp2_delta_logpower
Fp2_delta_logrelative
Fp2_gamma_logpower
Fp2_gamma_logrelative
Fp2_hjorth_complexity
Fp2_hjorth_mobility
Fp2_theta_logpower
Fp2_theta_logrelative
Fp2_total_logpower
Fz_alpha_logpower
Fz_alpha_logrelative
Fz_beta_logpower
Fz_beta_logrelative
Fz_delta_logpower
Fz_delta_logrelative
Fz_gamma_logpower
Fz_gamma_logrelative
Fz_hjorth_complexity
Fz_hjorth_mobility
Fz_theta_logpower
Fz_theta_logrelative
Fz_total_logpower
asymmetry_Fp2_minus_Fp1_alpha
asymmetry_Fp2_minus_Fp1_beta
asymmetry_Fp2_minus_Fp1_delta
asymmetry_Fp2_minus_Fp1_gamma
asymmetry_Fp2_minus_Fp1_theta
logratio_Fp1_Fz_alpha
logratio

In [5]:
print("=" * 60)
print("FEATURE SUMMARY")
print("=" * 60)

print(
    "All features      :",
    len(ALL_FEATURE_COLUMNS)
)

print(
    "Selected features :",
    len(SELECTED_FEATURES)
)

print()

print("Excluded features:")

excluded = [
    col
    for col in ALL_FEATURE_COLUMNS
    if col not in SELECTED_FEATURES
]

for col in excluded:
    print(col)

FEATURE SUMMARY
All features      : 108
Selected features : 54

Excluded features:
Fp1_delta_power
Fp1_theta_power
Fp1_alpha_power
Fp1_beta_power
Fp1_gamma_power
Fp1_delta_relative
Fp1_theta_relative
Fp1_alpha_relative
Fp1_beta_relative
Fp1_gamma_relative
Fp1_aperiodic_slope
Fp1_aperiodic_offset
Fp1_sample_entropy
Fp2_delta_power
Fp2_theta_power
Fp2_alpha_power
Fp2_beta_power
Fp2_gamma_power
Fp2_delta_relative
Fp2_theta_relative
Fp2_alpha_relative
Fp2_beta_relative
Fp2_gamma_relative
Fp2_aperiodic_slope
Fp2_aperiodic_offset
Fp2_sample_entropy
Fz_delta_power
Fz_theta_power
Fz_alpha_power
Fz_beta_power
Fz_gamma_power
Fz_delta_relative
Fz_theta_relative
Fz_alpha_relative
Fz_beta_relative
Fz_gamma_relative
Fz_aperiodic_slope
Fz_aperiodic_offset
Fz_sample_entropy
PLV_Fp1_Fp2_delta
PLV_Fp1_Fp2_theta
PLV_Fp1_Fp2_alpha
PLV_Fp1_Fp2_beta
PLV_Fp1_Fp2_gamma
PLV_Fp1_Fz_delta
PLV_Fp1_Fz_theta
PLV_Fp1_Fz_alpha
PLV_Fp1_Fz_beta
PLV_Fp1_Fz_gamma
PLV_Fp2_Fz_delta
PLV_Fp2_Fz_theta
PLV_Fp2_Fz_alpha
PLV_Fp2

In [6]:
# ============================================================
# CLEAN NUMERIC VALUES
# ============================================================

enrollment_X = enrollment_df[
    SELECTED_FEATURES
].copy()

verification_X = verification_df[
    SELECTED_FEATURES
].copy()


# Replace +/- infinity with NaN

enrollment_X = enrollment_X.replace(
    [np.inf, -np.inf],
    np.nan
)

verification_X = verification_X.replace(
    [np.inf, -np.inf],
    np.nan
)


print(
    "Enrollment NaNs:",
    enrollment_X.isna().sum().sum()
)

print(
    "Verification NaNs:",
    verification_X.isna().sum().sum()
)

Enrollment NaNs: 0
Verification NaNs: 0


In [7]:
# ============================================================
# MODEL
# ============================================================

def create_svm(C=10, gamma="scale"):

    model = Pipeline([
        
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "svm",
            SVC(
                kernel="rbf",
                C=C,
                gamma=gamma,
                probability=False,
                decision_function_shape="ovr",
                random_state=RANDOM_STATE
            )
        )
    ])

    return model

In [8]:
# ============================================================
# EER CALCULATION
# ============================================================

def calculate_eer(
    scores,
    genuine_labels,
    thresholds=None
):

    scores = np.asarray(scores)
    genuine_labels = np.asarray(
        genuine_labels,
        dtype=bool
    )

    impostor_labels = ~genuine_labels

    genuine_count = genuine_labels.sum()
    impostor_count = impostor_labels.sum()


    if thresholds is None:

        thresholds = np.linspace(
            scores.min(),
            scores.max(),
            2001
        )


    results = []


    for threshold in thresholds:

        accepted = scores >= threshold


        false_rejects = (
            (~accepted)
            & genuine_labels
        ).sum()


        false_accepts = (
            accepted
            & impostor_labels
        ).sum()


        frr = (
            false_rejects / genuine_count
            if genuine_count > 0
            else np.nan
        )


        far = (
            false_accepts / impostor_count
            if impostor_count > 0
            else np.nan
        )


        results.append({
            "threshold": threshold,
            "FAR": far,
            "FRR": frr,
            "TAR": 1 - frr,
            "difference": abs(far - frr)
        })


    results_df = pd.DataFrame(results)


    eer_row = results_df.loc[
        results_df["difference"].idxmin()
    ]


    eer = (
        eer_row["FAR"]
        + eer_row["FRR"]
    ) / 2


    return (
        eer,
        eer_row,
        results_df
    )

In [9]:
# ============================================================
# AUTHENTICATION SCORE GENERATION
# ============================================================

def create_authentication_trials(
    model,
    data_df,
    feature_columns
):

    X = data_df[
        feature_columns
    ]


    decision_scores = model.decision_function(X)


    classes = model.classes_


    # Make sure binary/multiclass shape is handled
    if decision_scores.ndim == 1:

        decision_scores = decision_scores.reshape(
            -1,
            1
        )


    trial_scores = []
    genuine_labels = []


    for i, (_, row) in enumerate(
        data_df.iterrows()
    ):

        true_subject = row["subject"]


        for class_idx, claimed_subject in enumerate(
            classes
        ):

            score = decision_scores[
                i,
                class_idx
            ]


            genuine = (
                claimed_subject
                == true_subject
            )


            trial_scores.append(score)
            genuine_labels.append(genuine)


    return (
        np.asarray(trial_scores),
        np.asarray(
            genuine_labels,
            dtype=bool
        )
    )

In [10]:
# ============================================================
# SESSION-WISE INTERNAL VALIDATION
# ============================================================

def evaluate_hyperparameters(
    C,
    gamma
):

    fold_eers = []


    # ========================================================
    # FOLD 1
    # Train S1 -> Validate S2
    # ========================================================

    train_df = enrollment_df[
        enrollment_df["session"]
        == "ses-session1"
    ]

    valid_df = enrollment_df[
        enrollment_df["session"]
        == "ses-session2"
    ]


    model = create_svm(
        C=C,
        gamma=gamma
    )


    model.fit(
        train_df[SELECTED_FEATURES],
        train_df["subject"]
    )


    scores, genuine = create_authentication_trials(
        model,
        valid_df,
        SELECTED_FEATURES
    )


    eer1, _, _ = calculate_eer(
        scores,
        genuine
    )


    fold_eers.append(eer1)


    # ========================================================
    # FOLD 2
    # Train S2 -> Validate S1
    # ========================================================

    train_df = enrollment_df[
        enrollment_df["session"]
        == "ses-session2"
    ]

    valid_df = enrollment_df[
        enrollment_df["session"]
        == "ses-session1"
    ]


    model = create_svm(
        C=C,
        gamma=gamma
    )


    model.fit(
        train_df[SELECTED_FEATURES],
        train_df["subject"]
    )


    scores, genuine = create_authentication_trials(
        model,
        valid_df,
        SELECTED_FEATURES
    )


    eer2, _, _ = calculate_eer(
        scores,
        genuine
    )


    fold_eers.append(eer2)


    mean_eer = np.mean(
        fold_eers
    )


    return (
        mean_eer,
        eer1,
        eer2
    )

In [11]:
# ============================================================
# HYPERPARAMETER SEARCH
# ============================================================

C_VALUES = [
    0.1,
    1,
    10,
    100,
    1000
]


GAMMA_VALUES = [
    "scale",
    0.0001,
    0.001,
    0.01,
    0.1
]


search_results = []


total_experiments = (
    len(C_VALUES)
    * len(GAMMA_VALUES)
)

experiment_number = 0


for C in C_VALUES:

    for gamma in GAMMA_VALUES:

        experiment_number += 1


        print(
            f"[{experiment_number}/{total_experiments}] "
            f"C={C}, gamma={gamma}"
        )


        mean_eer, eer_s1_s2, eer_s2_s1 = (
            evaluate_hyperparameters(
                C=C,
                gamma=gamma
            )
        )


        search_results.append({
            "C": C,
            "gamma": gamma,
            "mean_EER": mean_eer,
            "S1_to_S2_EER": eer_s1_s2,
            "S2_to_S1_EER": eer_s2_s1
        })


search_results_df = pd.DataFrame(
    search_results
)


search_results_df = search_results_df.sort_values(
    "mean_EER"
).reset_index(drop=True)


print()
print("=" * 70)
print("BEST HYPERPARAMETERS")
print("=" * 70)

print(
    search_results_df.head(10)
)

[1/25] C=0.1, gamma=scale


c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[2/25] C=0.1, gamma=0.0001


c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# ============================================================
# BEST PARAMETERS
# ============================================================

best_row = search_results_df.iloc[0]


BEST_C = best_row["C"]
BEST_GAMMA = best_row["gamma"]


print("=" * 60)
print("BEST PARAMETERS")
print("=" * 60)

print(
    "C     :", BEST_C
)

print(
    "Gamma :",
    BEST_GAMMA
)

print(
    "Mean validation EER :",
    f"{best_row['mean_EER'] * 100:.2f}%"
)

print(
    "S1 -> S2 EER        :",
    f"{best_row['S1_to_S2_EER'] * 100:.2f}%"
)

print(
    "S2 -> S1 EER        :",
    f"{best_row['S2_to_S1_EER'] * 100:.2f}%"
)

BEST PARAMETERS
C     : 1000.0
Gamma : 0.01
Mean validation EER : 10.45%
S1 -> S2 EER        : 10.85%
S2 -> S1 EER        : 10.04%


In [ ]:
# ============================================================
# FINAL MODEL
# TRAIN ON S1 + S2
# ============================================================

final_model = create_svm(
    C=BEST_C,
    gamma=BEST_GAMMA
)


final_model.fit(
    enrollment_df[SELECTED_FEATURES],
    enrollment_df["subject"]
)


print("Final model trained.")

print(
    "Training sessions:",
    ENROLL_SESSIONS
)

print(
    "Verification session:",
    VERIFY_SESSION
)

print(
    "Number of training samples:",
    len(enrollment_df)
)

print(
    "Number of subjects:",
    enrollment_df["subject"].nunique()
)

c:\Users\Sanjeev\OneDrive\Desktop\Brain ID\Model\EEG-Authentication\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Final model trained.
Training sessions: ['ses-session1', 'ses-session2']
Verification session: ses-session3
Number of training samples: 6100
Number of subjects: 60


In [ ]:
# ============================================================
# FINAL S3 VERIFICATION
# ============================================================

s3_scores, s3_genuine = (
    create_authentication_trials(
        final_model,
        verification_df,
        SELECTED_FEATURES
    )
)


print(
    "Total S3 trials:",
    len(s3_scores)
)

print(
    "Genuine trials:",
    s3_genuine.sum()
)

print(
    "Impostor trials:",
    (~s3_genuine).sum()
)

Total S3 trials: 158100
Genuine trials: 2635
Impostor trials: 155465


In [ ]:
# ============================================================
# FINAL EER
# ============================================================

final_eer, final_eer_row, final_metrics_df = (
    calculate_eer(
        s3_scores,
        s3_genuine
    )
)


print()
print("=" * 70)
print("FINAL S3 AUTHENTICATION RESULTS")
print("=" * 70)

print(
    f"EER       : {final_eer:.6f}"
)

print(
    f"EER %     : {final_eer * 100:.2f}%"
)

print(
    f"Threshold : {final_eer_row['threshold']:.6f}"
)

print(
    f"FAR       : {final_eer_row['FAR']:.6f}"
)

print(
    f"FRR       : {final_eer_row['FRR']:.6f}"
)

print(
    f"TAR       : {final_eer_row['TAR']:.6f}"
)

print("=" * 70)


FINAL S3 AUTHENTICATION RESULTS
EER       : 0.302062
EER %     : 30.21%
Threshold : 40.329112
FAR       : 0.299759
FRR       : 0.304364
TAR       : 0.695636


In [ ]:
# ============================================================
# COMPARISON
# ============================================================

OLD_BASELINE_EER = 0.246695

CURRENT_FULL_FEATURE_EER = 0.255192


improvement_vs_old = (
    OLD_BASELINE_EER
    - final_eer
)


improvement_vs_full = (
    CURRENT_FULL_FEATURE_EER
    - final_eer
)


print("=" * 70)
print("EER COMPARISON")
print("=" * 70)

print(
    f"Old baseline        : "
    f"{OLD_BASELINE_EER * 100:.2f}%"
)

print(
    f"Current full feature: "
    f"{CURRENT_FULL_FEATURE_EER * 100:.2f}%"
)

print(
    f"New compact model   : "
    f"{final_eer * 100:.2f}%"
)

print()

print(
    f"Improvement vs old baseline: "
    f"{improvement_vs_old * 100:.2f} percentage points"
)

print(
    f"Improvement vs full feature: "
    f"{improvement_vs_full * 100:.2f} percentage points"
)

print("=" * 70)

EER COMPARISON
Old baseline        : 24.67%
Current full feature: 25.52%
New compact model   : 30.21%

Improvement vs old baseline: -5.54 percentage points
Improvement vs full feature: -4.69 percentage points


In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

RESULTS_DIR = (
    DATA_ROOT
    / "model_results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Hyperparameter search
search_results_df.to_csv(
    RESULTS_DIR
    / "svm_hyperparameter_search.csv",
    index=False
)


# EER curve
final_metrics_df.to_csv(
    RESULTS_DIR
    / "s3_eer_curve.csv",
    index=False
)


# Configuration
config_df = pd.DataFrame({
    "parameter": [
        "enrollment_sessions",
        "verification_session",
        "selected_feature_count",
        "C",
        "gamma",
        "validation_EER",
        "S3_EER",
        "S3_FAR",
        "S3_FRR",
        "S3_TAR",
        "S3_threshold"
    ],

    "value": [
        ",".join(ENROLL_SESSIONS),
        VERIFY_SESSION,
        len(SELECTED_FEATURES),
        BEST_C,
        BEST_GAMMA,
        best_row["mean_EER"],
        final_eer,
        final_eer_row["FAR"],
        final_eer_row["FRR"],
        final_eer_row["TAR"],
        final_eer_row["threshold"]
    ]
})


config_df.to_csv(
    RESULTS_DIR
    / "experiment_configuration.csv",
    index=False
)


print(
    "Results saved to:"
)

print(
    RESULTS_DIR
)

Results saved to:
C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148\model_results


In [ ]:
# ============================================================
# PER-SUBJECT S3 PERFORMANCE
# ============================================================

def calculate_subject_scores(
    model,
    data_df,
    feature_columns
):

    X = data_df[
        feature_columns
    ]


    decision_scores = model.decision_function(X)

    classes = model.classes_


    rows = []


    for i, (_, row) in enumerate(
        data_df.iterrows()
    ):

        true_subject = row["subject"]


        for class_idx, claimed_subject in enumerate(
            classes
        ):

            score = decision_scores[
                i,
                class_idx
            ]


            genuine = (
                claimed_subject
                == true_subject
            )


            rows.append({
                "true_subject": true_subject,
                "claimed_subject": claimed_subject,
                "score": score,
                "genuine": genuine
            })


    return pd.DataFrame(rows)


subject_trials_df = calculate_subject_scores(
    final_model,
    verification_df,
    SELECTED_FEATURES
)


subject_results = []


for subject in sorted(
    verification_df["subject"].unique()
):

    subject_trials = subject_trials_df[
        subject_trials_df["true_subject"]
        == subject
    ]


    scores = subject_trials["score"].values

    genuine = subject_trials["genuine"].values


    if genuine.sum() == 0:
        continue

    if (~genuine).sum() == 0:
        continue


    eer, row, _ = calculate_eer(
        scores,
        genuine
    )


    subject_results.append({
        "subject": subject,
        "EER": eer,
        "EER_percent": eer * 100,
        "FAR": row["FAR"],
        "FRR": row["FRR"],
        "TAR": row["TAR"],
        "threshold": row["threshold"]
    })


subject_results_df = pd.DataFrame(
    subject_results
)


subject_results_df = subject_results_df.sort_values(
    "EER",
    ascending=False
)


print(
    subject_results_df.head(20)
)

   subject       EER  EER_percent       FAR       FRR       TAR  threshold
55  sub-59  0.812327    81.232666  0.806471  0.818182  0.181818  11.692089
38  sub-41  0.680226    68.022599  0.660452  0.700000  0.300000  20.701649
30  sub-32  0.666015    66.601478  0.665363  0.666667  0.333333  19.687499
24  sub-26  0.661259    66.125908  0.665375  0.657143  0.342857  19.836241
0   sub-01  0.648305    64.830508  0.653753  0.642857  0.357143  21.268593
14  sub-16  0.574576    57.457627  0.574153  0.575000  0.425000  26.190430
34  sub-37  0.543495    54.349452  0.542871  0.544118  0.455882  27.293762
49  sub-52  0.542857    54.285714  0.542857  0.542857  0.457143  26.785984
21  sub-23  0.539783    53.978343  0.524011  0.555556  0.444444  29.293159
26  sub-28  0.500000    50.000000  0.500000  0.500000  0.500000  29.261974
35  sub-38  0.481481    48.148148  0.481481  0.481481  0.518519  30.693850
16  sub-18  0.466384    46.638418  0.466102  0.466667  0.533333  32.334993
44  sub-47  0.432756    4